# Phi-3 Fine-tuning - Steel Plant Maintenance

Production-grade LoRA fine-tuning with optimized hyperparameters.

In [ ]:
# deps - uncomment if needed
# !pip install -q transformers datasets peft bitsandbytes wandb accelerate trl

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
import wandb

print(f'Torch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Config - Optimized Parameters

In [ ]:
# paths
TRAIN_FILE = '../data/training/train.jsonl'
VAL_FILE = '../data/training/val.jsonl'
OUTPUT_DIR = './phi3-steel-maintenance'

# model
MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'

# LoRA - optimized based on research
# Rank 32 for complex reasoning tasks (steel diagnostics)
# Alpha = 2*rank for stable training
LORA_R = 32  # higher for complex domain
LORA_ALPHA = 64  # 2x rank
LORA_DROPOUT = 0.1  # moderate dropout prevents overfitting
LORA_TARGET = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']  # all attention + FFN

# training - conservative for quality
EPOCHS = 3
BATCH_SIZE = 4  # larger batch for stability
GRAD_ACCUM = 2  # effective batch = 8
LR = 2e-4  # standard for LoRA
WARMUP_STEPS = 100  # warmup important for stability
MAX_SEQ_LEN = 2048

# optimization
WEIGHT_DECAY = 0.01  # regularization
MAX_GRAD_NORM = 1.0  # gradient clipping
LR_SCHEDULER = 'cosine'  # smooth decay

## Load Data

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(TRAIN_FILE)
val_data = load_jsonl(VAL_FILE)
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

In [ ]:
# inspect
s = train_data[0]
print('Keys:', list(s.keys()))
print('\nInstruction:', s['instruction'][:80])
print('Input:', s['input'][:120])
print('Output:', s['output'][:120])

## Format - Phi-3 Chat Template

In [ ]:
def format_prompt(sample):
    """phi-3 official chat format"""
    return f'''<|system|>
You are an expert AI system for predictive maintenance in steel manufacturing plants. Analyze equipment sensor data and provide detailed diagnostic assessments, root cause analysis, and actionable maintenance recommendations.<|end|>
<|user|>
{sample['instruction']}

{sample['input']}<|end|>
<|assistant|>
{sample['output']}<|end|>'''

print(format_prompt(train_data[0])[:400])

## Load Model - 4-bit Quantization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # required for training
print(f'Vocab size: {len(tokenizer):,}')

In [ ]:
# 4-bit quantization - NF4 is optimal for training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',  # normal float 4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # nested quantization
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2' if torch.cuda.is_available() else 'eager'
)

model.config.use_cache = False  # required for training
model.config.pretraining_tp = 1  # tensor parallelism
model = prepare_model_for_kbit_training(model)
print('Model loaded')

## LoRA Configuration

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Prepare Dataset

In [ ]:
train_ds = Dataset.from_list([{'text': format_prompt(s)} for s in train_data])
val_ds = Dataset.from_list([{'text': format_prompt(s)} for s in val_data])
print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

## Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    
    # optimization
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    optim='paged_adamw_32bit',
    
    # precision
    bf16=True,
    fp16=False,
    
    # logging & eval
    logging_strategy='steps',
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    
    # tracking - DUAL: WandB + TensorBoard
    report_to=['wandb', 'tensorboard'],  # both for comprehensive tracking
    logging_dir=f'{OUTPUT_DIR}/logs',  # TensorBoard logs
    run_name=f'phi3-steel-r{LORA_R}-{EPOCHS}ep',
    seed=42
)

print('Training config ready')
print(f'TensorBoard logs: {OUTPUT_DIR}/logs')
print(f'WandB project: steel-plant-maintenance')

## WandB Init

In [ ]:
# wandb.login()  # first time only

wandb.init(
    project='steel-plant-maintenance',
    name=f'phi3-r{LORA_R}-{EPOCHS}ep',
    config={
        'model': MODEL_NAME,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE * GRAD_ACCUM,
        'lr': LR,
        'train_samples': len(train_data),
        'val_samples': len(val_data)
    }
)

## Train with SFTTrainer

In [ ]:
# SFT (Supervised Fine-Tuning) Trainer from TRL
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False  # no packing for better quality
)

print('Starting training...')
trainer.train()

## Save Model

## View Training Metrics

In [ ]:
# View in TensorBoard
# Run in terminal: tensorboard --logdir=./phi3-steel-maintenance/logs --port=6006
# Then open: http://localhost:6006

# Or load inline (if in Colab/Jupyter)
# %load_ext tensorboard
# %tensorboard --logdir ./phi3-steel-maintenance/logs

print('Training complete! View metrics:')
print('1. WandB: https://wandb.ai/[your-project]')
print('2. TensorBoard: Run `tensorboard --logdir=./phi3-steel-maintenance/logs`')

In [ ]:
final_dir = f'{OUTPUT_DIR}/final'
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'Model saved to {final_dir}')

## Test Inference

In [ ]:
# test on validation sample
test_sample = val_data[5]
prompt = f'''<|system|>
You are an expert AI system for predictive maintenance in steel manufacturing plants.<|end|>
<|user|>
{test_sample['instruction']}

{test_sample['input']}<|end|>
<|assistant|>
'''

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('=== MODEL OUTPUT ===\n')
print(response[len(prompt):])  # print only generated part

In [ ]:
print('\n=== GROUND TRUTH ===\n')
print(test_sample['output'][:800])

In [ ]:
wandb.finish()
print('Training complete!')